In [1]:
import os
import pyspark
from pyspark.sql import SparkSession

# Dọn dẹp cache cũ
os.system("rm -rf ~/.ivy2/cache/*")
os.system("rm -rf ~/.ivy2/jars/*")

print("Đang khởi tạo Spark kết nối với MinIO và Iceberg (Ép tải lại từ Mirror)...")

# Khởi tạo Spark Session với bộ thư viện tương thích Spark 3.5
# Thêm tham số ivy.default.ivy.user.dir để ép Ivy tải lại sạch sẽ
spark = SparkSession.builder \
    .appName("Fraud_Detection_ML") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.extraJavaOptions", "-Divy.cache.ttl.default=eternal -Divy.home=/tmp/.ivy2") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.iceberg:iceberg-aws-bundle:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "rest") \
    .config("spark.sql.catalog.lakehouse.uri", "http://rest_catalog:8181") \
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

print("Spark đã khởi tạo thành công và sẵn sàng để sử dụng!")

Đang khởi tạo Spark kết nối với MinIO và Iceberg (Ép tải lại từ Mirror)...
Spark đã khởi tạo thành công và sẵn sàng để sử dụng!


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# =====================================================================
# NHIỆM VỤ A: KHÁM PHÁ ĐẶC TRƯNG (EDA) VÀ ĐỊNH NGHĨA VẤN ĐỀ
# =====================================================================
print("--- NHIỆM VỤ A: KHÁM PHÁ DỮ LIỆU (EDA) ---")

print("\n[1] Đang đọc dữ liệu từ MinIO (s3a://raw-data/creditcard.csv)...")
# Giả sử biến 'spark' đã được khởi tạo và file đã có trên MinIO
df_fraud = spark.read.csv("s3a://raw-data/creditcard.csv", header=True, inferSchema=True)

print("\n[2] Kiểm tra cấu trúc tập dữ liệu (Schema):")
# In ra một số cột đại diện (V1-V28, Time, Amount, Class)
df_fraud.select("Time", "V1", "V2", "Amount", "Class").printSchema()

print("\n[3] Tổng quan thống kê về số tiền giao dịch (Amount):")
df_fraud.groupBy("Class").agg(
    F.count("*").alias("Total_Transactions"),
    F.round(F.avg("Amount"), 2).alias("Avg_Amount"),
    F.round(F.max("Amount"), 2).alias("Max_Amount")
).show()

print("\n[4] PHÂN TÍCH SỰ MẤT CÂN BẰNG DỮ LIỆU:")
total_records = df_fraud.count()
fraud_count = df_fraud.filter(F.col("Class") == 1).count()
normal_count = total_records - fraud_count
fraud_percentage = (fraud_count / total_records) * 100

print(f"-> Tổng số giao dịch: {total_records:,}")
print(f"-> Giao dịch Bình thường (Class 0): {normal_count:,}")
print(f"-> Giao dịch Gian lận (Class 1)  : {fraud_count:,}")
print(f"-> Tỷ lệ Gian lận: {fraud_percentage:.3f}%")

print("\n[5] KIỂM TRA DỮ LIỆU KHUYẾT THIẾU (NULL/NAN):")
# Kiểm tra nhanh các cột quan trọng xem có dòng nào bị trống không
missing_check = df_fraud.select([
    F.count(F.when(F.col(c).isNull() | F.isnan(c), c)).alias(c) 
    for c in ["Time", "Amount", "Class"]
])
missing_check.show()

print("\n[6] PHÂN TÍCH HÀNH VI THEO KHUNG GIỜ (TIME ANALYSIS):")
# Chuyển đổi cột Time (giây) thành Giờ trong ngày (0-23h)
df_time = df_fraud.withColumn("Hour", (F.col("Time") / 3600).cast("integer") % 24)

# Tính tỷ lệ gian lận theo từng khung giờ
df_time.groupBy("Hour").agg(
    F.count("*").alias("Total_Tx"),
    F.sum("Class").alias("Fraud_Count")
).withColumn(
    "Fraud_Rate_Percent", 
    F.round((F.col("Fraud_Count") / F.col("Total_Tx")) * 100, 3)
).orderBy("Fraud_Rate_Percent", ascending=False).show(5)


# =====================================================================
# NHIỆM VỤ B: LỰA CHỌN THANG ĐO ĐÁNH GIÁ (EVALUATION METRICS)
# =====================================================================
print("\n--- NHIỆM VỤ B: THIẾT LẬP THANG ĐO (METRICS) ---")

print("Do tỷ lệ gian lận chỉ là ~0.17%, việc dùng Accuracy (Độ chính xác) là vô nghĩa.")
print("Chúng ta sẽ tập trung vào các thang đo sau:")

# 1. Đánh giá bằng ROC-AUC (Area Under ROC Curve)
# Thang đo mạnh mẽ để đánh giá khả năng phân tách giữa hai lớp (gian lận/bình thường).
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="Class", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
print("- ROC-AUC: Đã khởi tạo thành công.")

# 2. Đánh giá bằng Recall (Độ bao phủ)
# Tập trung đo lường: Trong TẤT CẢ các ca gian lận thực tế, mô hình bắt được bao nhiêu phần trăm?
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="Class", 
    predictionCol="prediction", 
    metricName="weightedRecall" 
)
print("- Recall: Đã khởi tạo thành công.")

# 3. Đánh giá bằng PR-AUC (Precision-Recall Area Under Curve)
# Cực kỳ quan trọng cho dữ liệu mất cân bằng nghiêm trọng.
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="Class", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderPR"
)
print("- PR-AUC: Đã khởi tạo thành công.")

# 4. Đánh giá bằng F1-Score
# Cân bằng giữa việc "không bỏ lọt gian lận" (Recall) và "không báo động giả quá nhiều" (Precision).
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="Class", 
    predictionCol="prediction", 
    metricName="f1" 
)
print("- F1-Score: Đã khởi tạo thành công.")

print("\n-> Tất cả Evaluator đã sẵn sàng để chấm điểm mô hình ở các bước tiếp theo!")

--- NHIỆM VỤ A: KHÁM PHÁ DỮ LIỆU (EDA) ---

[1] Đang đọc dữ liệu từ MinIO (s3a://raw-data/creditcard.csv)...

[2] Kiểm tra cấu trúc tập dữ liệu (Schema):
root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Class: integer (nullable = true)


[3] Tổng quan thống kê về số tiền giao dịch (Amount):
+-----+------------------+----------+----------+
|Class|Total_Transactions|Avg_Amount|Max_Amount|
+-----+------------------+----------+----------+
|    1|               492|    122.21|   2125.87|
|    0|            284315|     88.29|  25691.16|
+-----+------------------+----------+----------+


[4] PHÂN TÍCH SỰ MẤT CÂN BẰNG DỮ LIỆU:
-> Tổng số giao dịch: 284,807
-> Giao dịch Bình thường (Class 0): 284,315
-> Giao dịch Gian lận (Class 1)  : 492
-> Tỷ lệ Gian lận: 0.173%

[5] KIỂM TRA DỮ LIỆU KHUYẾT THIẾU (NULL/NAN):
+----+------+-----+
|Time|Amount|Class|
+----+------+-----+
|   0|     0|    0|
+---